<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Answer:

**One row = one content page.** The warehouse's daily table stores one row per page per day (report_date × client × page). For my lane, I add up each page's days into one row per page.

**Table:** `fact_content_daily_performance` (the daily table), March 2026 partition only (`month=2026-03`). I use only its Search Console columns.

**Time window:** 1–31 March 2026, split into two halves:
- **Feature window:** 1–15 March. This is everything my model is allowed to know.
- **Outcome window:** 16–31 March. Used only to build the proxy label.
- **Decision moment:** the morning of 16 March. At that moment an editor can see the first half, but the second half has not happened yet.

I use March because it is a mid-panel month, away from the final month (June 2026), which I keep sealed as a test month.

**What I rank (proxy label):** `still_under` = 1 if a page's CTR in 16–31 March was still below the typical CTR for its position band in that same period, otherwise 0. As in ML-03, this is a proxy: it shows whether low CTR lasted, not whether editing the page would help.

**Filter:** only rows where `gsc_data_available IS TRUE`, and only pages with at least 100 impressions in the first half and 50 in the second half, so that CTR is not just noise.

**Where this is verified:** in Section 3. Query 1 checks the grain (one row per page per day, no duplicates), Query 2 checks the row count and that the dates run 1–31 March 2026, and Query 3 checks how many rows survive the `gsc_data_available IS TRUE` filter.

In [10]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

# Read the token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB to the warehouse on Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# the daily table
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected. Working with: fact_content_daily_performance, month=2026-03")

Connected. Working with: fact_content_daily_performance, month=2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Answer:

**Features (clues), all from 1–15 March only:**
- `impressions_h1`: how many times the page was seen in search
- `ctr_h1`: clicks ÷ impressions × 100
- `position_h1`: average Google position, weighted by impressions
- `days_visible_h1`: on how many of the 15 days the page got at least one impression
- `missed_clicks_h1`: impressions × (typical CTR for its position band − its CTR) ÷ 100

**Label / proxy (the answer, never a clue):** `still_under`, built from `gsc_clicks`, `gsc_impressions` and `gsc_avg_position` in 16–31 March. Every second-half number (`clicks_h2`, `impressions_h2`, `ctr_h2`, `position_h2`) belongs here, not in the features.

**Context (for grouping, filtering and splitting only, never learned from):**
- `client_hash_id`, `content_hash_id`: scrambled IDs. Used to group days into pages and to test on clients the model has never seen.
- `report_date`: used only to split March into the two halves.
- `gsc_data_available`: used only as a filter (`IS TRUE`).

**Excluded (on purpose), with why:**
- **Any 16–31 March number as a feature.** This is my main deliberate exclusion: it is the outcome window, so using it would be seeing the future.
- **GA4 columns** (sessions, engagement, scroll, AI sessions): they measure what happens *after* someone clicks, so they cannot explain why people did not click. GA4 history also starts at different dates for different clients.
- **`gsc_sum_position`:** its exact meaning is not documented, so I compute position as an impressions-weighted average of `gsc_avg_position` instead.
- **June 2026 (the final month):** kept sealed as a future test month.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Answer:

Three queries on the March 2026 partition, each proving one claim from Sections 1–2:

**Query 1: grain.** Claim: one row of the daily table = one page on one day. Result: **0** duplicate (day, client, page) combinations, so the grain holds.

**Query 2: size and dates.** Claim: my slice is March 2026. Result: **9,841,378** rows covering **331,437** pages from **55** clients, running from **2026-03-01 to 2026-03-31** (31 days), with nothing outside March.

**Query 3: availability.** Claim: I keep only rows where search data is really present, filtering with `IS TRUE`. Result: **3,611,061** rows (**36.7%**) have `gsc_data_available` TRUE and survive the filter; **6,230,317** rows (63.3%) are FALSE and are dropped; **0** are empty in March.

What this tells me: almost two-thirds of the rows in this month carry no search data. Without the filter, those rows would add zeros that look like "nobody clicked" when really nothing was measured. The data dictionary warns that the flag can be empty in other months, so I filter with `IS TRUE` rather than `= TRUE` or `NOT FALSE`, which would mishandle empty values.

In [12]:
# QUERY 1 — GRAIN: is any page listed twice on the same day?
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate (day, client, page) combinations found:", len(dupes))
display(dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (day, client, page) combinations found: 0


,report_date,client_hash_id,content_hash_id,n_rows


In [13]:
# QUERY 2 — SIZE AND DATES: how big is my slice, and which days does it cover?
size = con.sql(f"""
    SELECT COUNT(*)                                                 AS total_rows,
           COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id)                           AS clients,
           MIN(report_date)                                         AS first_day,
           MAX(report_date)                                         AS last_day,
           COUNT(DISTINCT report_date)                              AS days
    FROM {MARCH}
""").df()

display(size)

,total_rows,pages,clients,first_day,last_day,days
0,9841378,331437,55,2026-03-01,2026-03-31,31


In [14]:
# QUERY 3 — AVAILABILITY: how many rows really have search data?
# The flag has THREE values (TRUE / FALSE / empty), so I filter with IS TRUE.
avail = con.sql(f"""
    SELECT COUNT(*)                                            AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS gsc_empty
    FROM {MARCH}
""").df()

avail["share_kept_pct"] = (avail["gsc_true"] / avail["all_rows"] * 100).round(1)
display(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,gsc_true,gsc_false,gsc_empty,share_kept_pct
0,9841378,3611061,6230317,0,36.7


### The five features (my feature frame)

For each page I add up its days in each half of March (only rows with `gsc_data_available IS TRUE`), keep pages with at least 100 impressions in 1–15 March and at least 50 in 16–31 March, and build five features from the **first half only**. Decision moment: the morning of 16 March.

| Feature | What it is | Knowable at the decision moment because… |
|---|---|---|
| `impressions_h1` | times the page was seen in search, 1–15 March | those impressions already happened before 16 March |
| `ctr_h1` | clicks ÷ impressions × 100, 1–15 March | both the clicks and impressions come from the finished first half |
| `position_h1` | average Google position, weighted by impressions, 1–15 March | Search Console reported these positions before 16 March |
| `days_visible_h1` | how many of the 15 days had at least one impression | it only counts days 1–15, which are all in the past at the decision moment |
| `missed_clicks_h1` | impressions × (typical CTR of its position band − its CTR) ÷ 100 | the page's numbers and the band's typical CTR are both computed only from 1–15 March |

The label `still_under` uses only 16–31 March, which has not happened yet at the decision moment.

In [15]:
# BUILD THE FEATURE FRAME: one row per page
import pandas as pd

pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        -- FIRST HALF (1-15 March): used for the features
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        -- SECOND HALF (16-31 March): used ONLY for the label
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]

# Features: first half only
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

# Label: second half only
pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

FEATURES = ["impressions_h1", "ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1"]

display(pages[["content_hash_id"] + FEATURES + ["still_under"]].head(10).round(2))
print("Rows (pages):", len(pages))
print("Unique pages:", len(pages[["client_hash_id", "content_hash_id"]].drop_duplicates()))
print("Clients:", pages["client_hash_id"].nunique())
print("Share still under-performing in 16-31 March:", round(pages["still_under"].mean(), 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_h1,ctr_h1,position_h1,days_visible_h1,missed_clicks_h1,still_under
0,content_2e6360ad20fd7107,219.0,0.46,4.44,15,-0.46,1
1,content_d49a012dcb924e31,246.0,0.00,4.81,15,0.60,1
2,content_614baf2af4330bd7,413.0,0.24,4.66,15,0.01,1
3,content_225dc9235023be5f,279.0,0.36,10.85,15,-0.75,0
4,content_babcf791dccc1610,101.0,0.00,7.14,15,0.14,0
5,content_26f5092ee7f70d45,2553.0,0.00,7.51,15,3.62,1
6,content_cfad137c1b04251b,183.0,0.00,6.85,15,0.30,1
7,content_9e7c70abfbae371e,1676.0,0.00,5.66,15,2.78,1
8,content_c0fc3b40ce00a5d7,166.0,0.60,1.57,15,-0.59,1
9,content_d79c0df3e1b437fb,167.0,0.00,1.82,15,0.41,1


Rows (pages): 75817
Unique pages: 75817
Clients: 37
Share still under-performing in 16-31 March: 0.35


In [16]:
# HONEST QUICK SCORE: train on some clients, test on clients the tree has never seen
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

def quick_score(feature_list):
    split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(split.split(pages, groups=pages["client_hash_id"]))
    train, test = pages.iloc[train_idx], pages.iloc[test_idx]
    tree = DecisionTreeClassifier(max_depth=4, random_state=42)
    tree.fit(train[feature_list], train["still_under"])
    accuracy = (tree.predict(test[feature_list]) == test["still_under"]).mean()
    lazy_guess = max(test["still_under"].mean(), 1 - test["still_under"].mean())
    return accuracy, lazy_guess

honest, lazy = quick_score(FEATURES)
print(f"Lazy guess (always say the most common answer): {lazy:.2f}")
print(f"Honest tree, 5 features:                        {honest:.2f}")

Lazy guess (always say the most common answer): 0.58
Honest tree, 5 features:                        0.70


In [17]:
# THE TRAP: add ONE column built from the label on purpose
# ctr_gap_h2 = how far below typical the page was in 16-31 March.
# still_under is just "ctr_gap_h2 > 0", so this column IS the answer in disguise.
pages["ctr_gap_h2"] = pages["expected_ctr_h2"] - pages["ctr_h2"]

leaky, _ = quick_score(FEATURES + ["ctr_gap_h2"])
print(f"With the leaked column:  {leaky:.2f}   <- too good to be true")

# Remove it, and keep the honest number
pages = pages.drop(columns=["ctr_gap_h2"])
honest_again, _ = quick_score(FEATURES)
print(f"Leak removed:            {honest_again:.2f}   <- the honest number I keep")
print("Is ctr_gap_h2 still in my data?", "ctr_gap_h2" in pages.columns)

With the leaked column:  1.00   <- too good to be true
Leak removed:            0.70   <- the honest number I keep
Is ctr_gap_h2 still in my data? False


### The trap: a deliberate leak

I added one label-derived column on purpose: `ctr_gap_h2` = the typical CTR for the page's position band in 16–31 March minus the page's own CTR in 16–31 March. My label `still_under` is simply "`ctr_gap_h2` > 0", so this column is the answer in disguise. It also comes from the outcome window, which has not happened yet at the decision moment.

| Version | Accuracy on clients the tree never saw |
|---|---|
| Lazy guess (always the most common answer) | 0.58 |
| Honest tree, 5 first-half features | **0.70** |
| Tree + leaked `ctr_gap_h2` | 1.00 |
| Leak removed | **0.70** (the number I keep) |

With the leak, the tree scores a perfect 1.00. That is not skill: it simply reads the answer from the leaked column. After I delete the column, the score returns to 0.70. That is the honest number: 12 points above the lazy guess, measured on clients the tree never saw during training.

Lesson: when a score suddenly looks perfect, the first suspect is a feature built from the label or from the future.

These scores come from one train/test split (random_state=42) and a depth-4 tree, so they are a quick check, not a final evaluation.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Answer:

**My named limitation: coverage.** My feature frame covers only **75,817 of the 331,437 pages (23%)** and **37 of the 55 clients** in March 2026 (numbers below). Two things shrink it: most rows have no search data (only 36.7% have `gsc_data_available` TRUE), and I keep only pages with at least 100 impressions in 1–15 March and 50 in 16–31 March. So my results describe busier, well-measured pages from 37 clients. They do not describe quiet pages or the other 18 clients, and any finding should be stated for that group only.

Smaller limits worth knowing:
- Each half of March is only about two weeks, and one month says nothing about seasonality.
- The proxy shows whether low CTR lasted for two weeks, not whether editing the page would help. This is decision-support, not proof of cause.
- In a position band where the typical CTR is 0 (common far down the results), a page cannot be "below typical", so those pages always get `still_under` = 0.

In [18]:
# DATA LIMIT: how much of March does my feature frame actually cover?
# (Uses the results already loaded above, so this is not a new query.)
march_pages   = size["pages"][0]
march_clients = size["clients"][0]
frame_pages   = len(pages)
frame_clients = pages["client_hash_id"].nunique()

print(f"Pages:   {frame_pages:,} of {march_pages:,} in March ({frame_pages / march_pages * 100:.0f}%)")
print(f"Clients: {frame_clients} of {march_clients} in March")

Pages:   75,817 of 331,437 in March (23%)
Clients: 37 of 55 in March


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.